# 02 — Data Cleaning & Transformation Pipeline

**Project**: Hospital Readmission Rate Analysis (Diabetes 130-US Hospitals)  
**Team**: SEC-A, Group 3  
**Sector**: Healthcare  

---

## Objective

This notebook implements the **Transform** phase of the ETL pipeline. Every transformation step is logged with:
- **What** was changed
- **Why** it was changed
- **Row/column impact** (before → after counts)

### Cleaning Strategy (based on 01_extraction.ipynb findings)

| Step | Action | Rationale |
|------|--------|----------|
| 1 | Replace `?` with `NaN` | Standardise missing value representation |
| 2 | Drop `weight` column | 96.9% missing — unusable |
| 3 | Drop `payer_code` column | 39.6% missing, not relevant to readmission |
| 4 | Drop near-zero variance medication columns | >99% `No` — no analytical signal |
| 5 | Drop `gender = Unknown/Invalid` rows | Only 3 rows — negligible loss |
| 6 | Deduplicate by `patient_nbr` | Keep first encounter to prevent data leakage |
| 7 | Handle `medical_specialty` missing | Recode to `Other` |
| 8 | Handle `race` missing | Recode to `Unknown` |
| 9 | Map IDs to meaningful labels | admission_type, discharge_disposition, admission_source |
| 10 | Standardise `age` to numeric midpoints | For statistical analysis |
| 11 | Recode lab results | `None` → `Not Measured` |
| 12 | Map ICD-9 diagnosis codes to categories | Group into clinical families |
| 13 | Feature engineering | Total visits, medication count, binary target |
| 14 | Validate & export | Save to `data/processed/` |

> **Input**: `data/raw/RAW_HEALTHCARE.csv`  
> **Output**: `data/processed/cleaned_healthcare.csv`

## 1. Environment Setup & Data Load

In [1]:
# ============================================================
# 02_cleaning.ipynb — Environment Setup
# ============================================================

import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 55)

# Cleaning log to track every transformation
cleaning_log = []

def log_step(step_num, action, rows_before, rows_after, cols_before, cols_after, details=""):
    """Log each cleaning step with before/after metrics."""
    entry = {
        'step': step_num,
        'action': action,
        'rows_before': rows_before,
        'rows_after': rows_after,
        'rows_removed': rows_before - rows_after,
        'cols_before': cols_before,
        'cols_after': cols_after,
        'cols_removed': cols_before - cols_after,
        'details': details
    }
    cleaning_log.append(entry)
    print(f"  Step {step_num}: {action}")
    print(f"    Rows: {rows_before:,} → {rows_after:,} (Δ {rows_before - rows_after:,})")
    print(f"    Cols: {cols_before} → {cols_after} (Δ {cols_before - cols_after})")
    if details:
        print(f"    Note: {details}")
    print()

print(f"Cleaning pipeline started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Cleaning pipeline started at: 2026-04-21 13:50:41


In [2]:
# ============================================================
# Load raw dataset (preserving '?' as string for explicit handling)
# ============================================================

RAW_DATA_PATH = os.path.join('..', 'data', 'raw', 'RAW_HEALTHCARE.csv')
PROCESSED_DATA_PATH = os.path.join('..', 'data', 'processed', 'cleaned_healthcare.csv')

df = pd.read_csv(RAW_DATA_PATH, na_values=[], keep_default_na=False)
print(f"✅ Raw data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Columns: {list(df.columns)}")

✅ Raw data loaded: 101,766 rows × 50 columns
   Columns: ['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


## 2. Step 1 — Replace `?` Placeholders with NaN

In [3]:
# ============================================================
# Step 1: Replace '?' with NaN across all columns
# Rationale: '?' is a non-standard missing indicator. Standardising
# to NaN enables pandas-native missing value handling.
# ============================================================

rows_b, cols_b = df.shape

# Count before replacement
q_total = (df == '?').sum().sum()
print(f"Total '?' values found: {q_total:,}")

df.replace('?', np.nan, inplace=True)

rows_a, cols_a = df.shape
log_step(1, "Replace '?' with NaN", rows_b, rows_a, cols_b, cols_a,
         f"{q_total:,} placeholder values converted")

Total '?' values found: 192,849
  Step 1: Replace '?' with NaN
    Rows: 101,766 → 101,766 (Δ 0)
    Cols: 50 → 50 (Δ 0)
    Note: 192,849 placeholder values converted



## 3. Step 2 — Drop High-Missing Columns

In [4]:
# ============================================================
# Step 2: Drop columns with excessive missing values
# - weight:     96.9% missing — no meaningful imputation possible
# - payer_code: 39.6% missing — insurance payer is not central
#              to the clinical readmission question
# Using errors='ignore' to make this cell safe to re-run.
# ============================================================

rows_b, cols_b = df.shape

drop_high_missing = [c for c in ['weight', 'payer_code'] if c in df.columns]
if drop_high_missing:
    df.drop(columns=drop_high_missing, inplace=True)
    print(f"Dropped columns: {drop_high_missing}")
else:
    print("Columns already dropped (safe re-run).")

rows_a, cols_a = df.shape
log_step(2, "Drop high-missing columns (weight, payer_code)",
         rows_b, rows_a, cols_b, cols_a,
         "weight 96.9% missing, payer_code 39.6% missing")

Dropped columns: ['weight', 'payer_code']
  Step 2: Drop high-missing columns (weight, payer_code)
    Rows: 101,766 → 101,766 (Δ 0)
    Cols: 50 → 48 (Δ 2)
    Note: weight 96.9% missing, payer_code 39.6% missing



## 4. Step 3 — Drop Near-Zero Variance Medication Columns

In [5]:
# ============================================================
# Step 3: Remove medication columns where >99% of values are 'No'
# These columns contain almost no variation and add noise, not signal.
# ============================================================

all_med_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
    'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

rows_b, cols_b = df.shape

low_var_meds = []
for col in all_med_cols:
    if col in df.columns:
        no_pct = (df[col] == 'No').mean() * 100
        if no_pct > 99:
            low_var_meds.append(col)

if low_var_meds:
    print(f"Near-zero variance medications to drop ({len(low_var_meds)}): {low_var_meds}")
    df.drop(columns=low_var_meds, inplace=True)
else:
    print("No near-zero variance medications found (already dropped or all are informative).")

rows_a, cols_a = df.shape
log_step(3, f"Drop {len(low_var_meds)} near-zero variance medication columns",
         rows_b, rows_a, cols_b, cols_a,
         f"Dropped: {low_var_meds}")

Near-zero variance medications to drop (15): ['nateglinide', 'chlorpropamide', 'acetohexamide', 'tolbutamide', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
  Step 3: Drop 15 near-zero variance medication columns
    Rows: 101,766 → 101,766 (Δ 0)
    Cols: 48 → 33 (Δ 15)
    Note: Dropped: ['nateglinide', 'chlorpropamide', 'acetohexamide', 'tolbutamide', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']



## 5. Step 4 — Remove Invalid Gender Rows

In [6]:
# ============================================================
# Step 4: Drop rows with gender = 'Unknown/Invalid'
# Only 3 records — negligible loss, ensures clean binary gender field
# ============================================================

rows_b, cols_b = df.shape

n_invalid = (df['gender'] == 'Unknown/Invalid').sum()
df = df[df['gender'] != 'Unknown/Invalid'].copy()

rows_a, cols_a = df.shape
log_step(4, "Drop rows with Unknown/Invalid gender",
         rows_b, rows_a, cols_b, cols_a,
         f"{n_invalid} rows removed")

  Step 4: Drop rows with Unknown/Invalid gender
    Rows: 101,766 → 101,763 (Δ 3)
    Cols: 33 → 33 (Δ 0)
    Note: 3 rows removed



## 6. Step 5 — Deduplicate by Patient

Multiple encounters from the same patient can cause **data leakage** — earlier encounters can carry information about future readmissions. We keep only the **first encounter** per patient.

In [7]:
# ============================================================
# Step 5: Keep only the first encounter per patient
# Rationale: Prevents information leakage from repeated patient visits
# ============================================================

rows_b, cols_b = df.shape

df.sort_values('encounter_id', inplace=True)
df.drop_duplicates(subset='patient_nbr', keep='first', inplace=True)

rows_a, cols_a = df.shape
log_step(5, "Deduplicate by patient_nbr (keep first encounter)",
         rows_b, rows_a, cols_b, cols_a,
         f"{rows_b - rows_a:,} duplicate patient encounters removed")

  Step 5: Deduplicate by patient_nbr (keep first encounter)
    Rows: 101,763 → 71,515 (Δ 30,248)
    Cols: 33 → 33 (Δ 0)
    Note: 30,248 duplicate patient encounters removed



## 7. Step 6 — Handle Missing Values in Remaining Columns

In [8]:
# ============================================================
# Step 6a: medical_specialty — fill NaN with 'Other'
# 49% missing, but specialty is valuable for segmentation.
# ============================================================

rows_b, cols_b = df.shape

n_missing_spec = df['medical_specialty'].isna().sum()
df['medical_specialty'] = df['medical_specialty'].fillna('Other')

print(f"  medical_specialty: {n_missing_spec:,} NaN → filled with 'Other'")

  medical_specialty: 34,475 NaN → filled with 'Other'


In [9]:
# ============================================================
# Step 6b: race — fill NaN with 'Unknown'
# Only 2.2% missing, low impact but clean representation.
# ============================================================

n_missing_race = df['race'].isna().sum()
df['race'] = df['race'].fillna('Unknown')

print(f"  race: {n_missing_race:,} NaN → filled with 'Unknown'")

  race: 1,946 NaN → filled with 'Unknown'


In [10]:
# ============================================================
# Step 6c: Diagnosis codes — fill NaN with 'Unknown'
# ============================================================

for diag_col in ['diag_1', 'diag_2', 'diag_3']:
    n_miss = df[diag_col].isna().sum()
    df[diag_col] = df[diag_col].fillna('Unknown')
    print(f"  {diag_col}: {n_miss:,} NaN → filled with 'Unknown'")

rows_a, cols_a = df.shape
log_step(6, "Handle missing values (medical_specialty, race, diagnoses)",
         rows_b, rows_a, cols_b, cols_a,
         "NaN filled with domain-appropriate labels")

  diag_1: 11 NaN → filled with 'Unknown'
  diag_2: 294 NaN → filled with 'Unknown'
  diag_3: 1,225 NaN → filled with 'Unknown'
  Step 6: Handle missing values (medical_specialty, race, diagnoses)
    Rows: 71,515 → 71,515 (Δ 0)
    Cols: 33 → 33 (Δ 0)
    Note: NaN filled with domain-appropriate labels



## 8. Step 7 — Map Coded IDs to Meaningful Labels

In [11]:
# ============================================================
# Step 7: Map admission_type_id to descriptive labels
# Source: ICD-9 coding reference
# ============================================================

rows_b, cols_b = df.shape

# Only process if original ID columns still exist
id_cols_present = [c for c in ['admission_type_id', 'discharge_disposition_id', 'admission_source_id'] if c in df.columns]

if id_cols_present:
    # Convert to int first (may be object after NaN handling)
    for col in id_cols_present:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    admission_type_map = {
        1: 'Emergency', 2: 'Urgent', 3: 'Elective', 4: 'Newborn',
        5: 'Not Available', 6: 'Not Mapped', 7: 'Trauma Center', 8: 'Not Mapped'
    }

    discharge_map = {
        1: 'Discharged to Home', 2: 'Short-term Hospital Transfer',
        3: 'SNF (Skilled Nursing)', 4: 'ICF (Intermediate Care)',
        5: 'Other Institution Transfer', 6: 'Home Health Service',
        7: 'AMA (Against Medical Advice)', 8: 'Home IV Service',
        9: 'Admitted as Inpatient', 10: 'Neonate Discharge',
        11: 'Expired', 12: 'Still Patient',
        13: 'Hospice / Home', 14: 'Hospice / Medical Facility',
        15: 'Swing Bed', 16: 'Outpatient Services',
        17: 'Undetermined', 18: 'NULL',
        19: 'Expired (at home)', 20: 'Expired (medical facility)',
        21: 'Expired (place unknown)', 22: 'Rehab Facility Transfer',
        23: 'Long-term Hospital Transfer', 24: 'Medicaid Facility',
        25: 'Not Mapped', 26: 'Unknown',
        27: 'Psych Hospital Transfer', 28: 'Critical Access Hospital',
        29: 'Federal Health Facility', 30: 'Not Mapped'
    }

    admission_source_map = {
        1: 'Physician Referral', 2: 'Clinic Referral',
        3: 'HMO Referral', 4: 'Hospital Transfer',
        5: 'SNF Transfer', 6: 'Other Facility Transfer',
        7: 'Emergency Room', 8: 'Court/Law Enforcement',
        9: 'Not Available', 10: 'Extramural Birth',
        11: 'Normal Delivery', 12: 'Premature Delivery',
        13: 'Sick Baby', 14: 'Extramural Birth',
        15: 'Not Available', 17: 'NULL',
        20: 'Not Mapped', 21: 'Not Mapped',
        22: 'Not Mapped', 25: 'Not Mapped', 26: 'Hospital Transfer'
    }

    if 'admission_type_id' in df.columns:
        df['admission_type'] = df['admission_type_id'].map(admission_type_map).fillna('Other')
    if 'discharge_disposition_id' in df.columns:
        df['discharge_disposition'] = df['discharge_disposition_id'].map(discharge_map).fillna('Other')
    if 'admission_source_id' in df.columns:
        df['admission_source'] = df['admission_source_id'].map(admission_source_map).fillna('Other')

    # Drop the original ID columns
    df.drop(columns=id_cols_present, inplace=True)
    print(f"Mapped and dropped ID columns: {id_cols_present}")
else:
    print("ID columns already mapped (safe re-run).")

rows_a, cols_a = df.shape
log_step(7, "Map coded IDs to descriptive labels",
         rows_b, rows_a, cols_b, cols_a,
         "admission_type, discharge_disposition, admission_source created; original ID columns dropped")

Mapped and dropped ID columns: ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
  Step 7: Map coded IDs to descriptive labels
    Rows: 71,515 → 71,515 (Δ 0)
    Cols: 33 → 33 (Δ 0)
    Note: admission_type, discharge_disposition, admission_source created; original ID columns dropped



## 9. Step 8 — Standardise Age to Numeric Midpoints

In [12]:
# ============================================================
# Step 8: Convert age ranges to numeric midpoints
# Rationale: Enables statistical analysis (correlation, regression)
# while preserving the original binned representation as well.
# ============================================================

rows_b, cols_b = df.shape

age_midpoint_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}

# Use 'age' column if it exists, otherwise age_group is already set
if 'age' in df.columns:
    df['age_group'] = df['age']
    df['age_midpoint'] = df['age'].map(age_midpoint_map)
    df.drop(columns=['age'], inplace=True)
    print("Age mapping applied from 'age' column.")
elif 'age_group' in df.columns and 'age_midpoint' not in df.columns:
    df['age_midpoint'] = df['age_group'].map(age_midpoint_map)
    print("Age mapping applied from 'age_group' column.")
else:
    print("Age already processed (safe re-run).")

print("Age mapping:")
if 'age_group' in df.columns and 'age_midpoint' in df.columns:
    print(df[['age_group', 'age_midpoint']].drop_duplicates().sort_values('age_midpoint').to_string(index=False))

rows_a, cols_a = df.shape
log_step(8, "Standardise age to numeric midpoints",
         rows_b, rows_a, cols_b, cols_a,
         "age_group (original) + age_midpoint (numeric) created")

Age mapping applied from 'age' column.
Age mapping:
age_group  age_midpoint
   [0-10)             5
  [10-20)            15
  [20-30)            25
  [30-40)            35
  [40-50)            45
  [50-60)            55
  [60-70)            65
  [70-80)            75
  [80-90)            85
 [90-100)            95
  Step 8: Standardise age to numeric midpoints
    Rows: 71,515 → 71,515 (Δ 0)
    Cols: 33 → 34 (Δ -1)
    Note: age_group (original) + age_midpoint (numeric) created



## 10. Step 9 — Recode Lab Results

In [13]:
# ============================================================
# Step 9: Recode lab result columns
# 'None' in max_glu_serum and A1Cresult means 'Not Measured'
# — this is clinically different from a normal result.
# ============================================================

rows_b, cols_b = df.shape

df['max_glu_serum'] = df['max_glu_serum'].replace('None', 'Not Measured')
df['A1Cresult'] = df['A1Cresult'].replace('None', 'Not Measured')

# Create binary flags: was the test performed?
df['glu_test_performed'] = (df['max_glu_serum'] != 'Not Measured').astype(int)
df['a1c_test_performed'] = (df['A1Cresult'] != 'Not Measured').astype(int)

print("max_glu_serum values:")
print(df['max_glu_serum'].value_counts())
print(f"\nA1Cresult values:")
print(df['A1Cresult'].value_counts())

rows_a, cols_a = df.shape
log_step(9, "Recode lab results (None → Not Measured) + binary flags",
         rows_b, rows_a, cols_b, cols_a,
         "glu_test_performed and a1c_test_performed flags added")

max_glu_serum values:
max_glu_serum
Not Measured    68059
Norm             1731
>200              969
>300              756
Name: count, dtype: int64

A1Cresult values:
A1Cresult
Not Measured    58529
>8               6304
Norm             3791
>7               2891
Name: count, dtype: int64
  Step 9: Recode lab results (None → Not Measured) + binary flags
    Rows: 71,515 → 71,515 (Δ 0)
    Cols: 34 → 36 (Δ -2)
    Note: glu_test_performed and a1c_test_performed flags added



## 11. Step 10 — Map ICD-9 Diagnosis Codes to Clinical Categories

In [14]:
# ============================================================
# Step 10: Group ICD-9 diagnosis codes into clinical families
# Rationale: Raw codes are too granular (>700 unique). Grouping
# into standard ICD-9 chapters makes analysis meaningful.
# Reference: https://en.wikipedia.org/wiki/List_of_ICD-9_codes
# ============================================================

rows_b, cols_b = df.shape

def map_icd9_to_category(code):
    """Map ICD-9 diagnosis code to a clinical category."""
    if code == 'Unknown' or pd.isna(code):
        return 'Unknown'
    code_str = str(code).strip()
    if code_str.startswith('V'):
        return 'Supplementary (V-codes)'
    if code_str.startswith('E'):
        return 'External Causes (E-codes)'
    try:
        num_code = float(code_str)
    except ValueError:
        return 'Other'
    if 1 <= num_code <= 139:
        return 'Infectious & Parasitic'
    elif 140 <= num_code <= 239:
        return 'Neoplasms'
    elif 240 <= num_code <= 279:
        return 'Endocrine/Metabolic (incl. Diabetes)'
    elif 280 <= num_code <= 289:
        return 'Blood & Immune'
    elif 290 <= num_code <= 319:
        return 'Mental Disorders'
    elif 320 <= num_code <= 389:
        return 'Nervous System & Sense Organs'
    elif 390 <= num_code <= 459:
        return 'Circulatory System'
    elif 460 <= num_code <= 519:
        return 'Respiratory System'
    elif 520 <= num_code <= 579:
        return 'Digestive System'
    elif 580 <= num_code <= 629:
        return 'Genitourinary System'
    elif 630 <= num_code <= 679:
        return 'Pregnancy & Childbirth'
    elif 680 <= num_code <= 709:
        return 'Skin & Subcutaneous'
    elif 710 <= num_code <= 739:
        return 'Musculoskeletal'
    elif 740 <= num_code <= 759:
        return 'Congenital Anomalies'
    elif 760 <= num_code <= 779:
        return 'Perinatal Conditions'
    elif 780 <= num_code <= 799:
        return 'Symptoms & Ill-defined'
    elif 800 <= num_code <= 999:
        return 'Injury & Poisoning'
    else:
        return 'Other'

# Apply mapping to all three diagnosis columns
df['diag_1_category'] = df['diag_1'].apply(map_icd9_to_category)
df['diag_2_category'] = df['diag_2'].apply(map_icd9_to_category)
df['diag_3_category'] = df['diag_3'].apply(map_icd9_to_category)

# Check if primary diagnosis is diabetes-related (250.xx)
df['primary_diag_diabetes'] = df['diag_1'].apply(
    lambda x: 1 if str(x).startswith('250') else 0
)

print("Primary diagnosis category distribution:")
print(df['diag_1_category'].value_counts().head(10))

rows_a, cols_a = df.shape
log_step(10, "Map ICD-9 diagnosis codes to clinical categories",
         rows_b, rows_a, cols_b, cols_a,
         "diag_1/2/3_category + primary_diag_diabetes created")

Primary diagnosis category distribution:
diag_1_category
Circulatory System                      21820
Endocrine/Metabolic (incl. Diabetes)     7685
Respiratory System                       6736
Digestive System                         6403
Symptoms & Ill-defined                   5530
Injury & Poisoning                       4777
Musculoskeletal                          4080
Genitourinary System                     3488
Neoplasms                                2742
Infectious & Parasitic                   1818
Name: count, dtype: int64
  Step 10: Map ICD-9 diagnosis codes to clinical categories
    Rows: 71,515 → 71,515 (Δ 0)
    Cols: 36 → 40 (Δ -4)
    Note: diag_1/2/3_category + primary_diag_diabetes created



## 12. Step 11 — Feature Engineering

In [15]:
# ============================================================
# Step 11: Create derived features for analysis
# ============================================================

rows_b, cols_b = df.shape

# --- 11a. Total prior visits (outpatient + emergency + inpatient) ---
df['total_prior_visits'] = (
    pd.to_numeric(df['number_outpatient'], errors='coerce').fillna(0) +
    pd.to_numeric(df['number_emergency'], errors='coerce').fillna(0) +
    pd.to_numeric(df['number_inpatient'], errors='coerce').fillna(0)
).astype(int)

# --- 11b. Number of active medications (non-'No' medication columns) ---
remaining_med_cols = [col for col in all_med_cols if col in df.columns]
df['n_active_medications'] = df[remaining_med_cols].apply(
    lambda row: (row != 'No').sum(), axis=1
)

# --- 11c. Number of medication changes (Up or Down) ---
df['n_med_changes'] = df[remaining_med_cols].apply(
    lambda row: row.isin(['Up', 'Down']).sum(), axis=1
)

# --- 11d. Binary target: readmitted within 30 days ---
df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)

# --- 11e. Categorised readmission (for Tableau) ---
df['readmission_status'] = df['readmitted'].map({
    '<30': 'Readmitted <30 Days',
    '>30': 'Readmitted >30 Days',
    'NO': 'Not Readmitted'
})

# --- 11f. Medication change flag (binary) ---
df['med_changed'] = (df['change'] == 'Ch').astype(int)

# --- 11g. Diabetes medication flag (binary) ---
df['diabetes_med_flag'] = (df['diabetesMed'] == 'Yes').astype(int)

# --- 11h. Comorbidity indicator: number of distinct diagnosis categories ---
df['n_distinct_diag_categories'] = df[['diag_1_category', 'diag_2_category', 'diag_3_category']].nunique(axis=1)

# --- 11i. High-utiliser flag: patients with >3 total prior visits ---
df['high_utiliser'] = (df['total_prior_visits'] > 3).astype(int)

rows_a, cols_a = df.shape
log_step(11, "Feature engineering",
         rows_b, rows_a, cols_b, cols_a,
         "total_prior_visits, n_active_medications, n_med_changes, readmitted_binary, "
         "readmission_status, med_changed, diabetes_med_flag, n_distinct_diag_categories, high_utiliser")

print(f"\nNew features created: {cols_a - cols_b}")

  Step 11: Feature engineering
    Rows: 71,515 → 71,515 (Δ 0)
    Cols: 40 → 49 (Δ -9)
    Note: total_prior_visits, n_active_medications, n_med_changes, readmitted_binary, readmission_status, med_changed, diabetes_med_flag, n_distinct_diag_categories, high_utiliser


New features created: 9


## 13. Step 12 — Consolidate Medical Specialty

There are ~70+ unique specialities. We group the less frequent ones to prevent sparsity.

In [16]:
# ============================================================
# Step 12: Group rare specialties
# Keep top specialties by volume; group the rest as 'Other'
# ============================================================

rows_b, cols_b = df.shape

# Keep specialties with at least 1% of encounters
specialty_counts = df['medical_specialty'].value_counts()
threshold = len(df) * 0.01  # 1% threshold
top_specialties = specialty_counts[specialty_counts >= threshold].index.tolist()

print(f"Specialties above 1% threshold ({len(top_specialties)}): {top_specialties}")
print(f"Specialties grouped into 'Other': {len(specialty_counts) - len(top_specialties)}")

df['medical_specialty_grouped'] = df['medical_specialty'].where(
    df['medical_specialty'].isin(top_specialties), 'Other'
)

rows_a, cols_a = df.shape
log_step(12, "Group rare medical specialties into 'Other'",
         rows_b, rows_a, cols_b, cols_a,
         f"Kept {len(top_specialties)} specialties, grouped rest as 'Other'")

Specialties above 1% threshold (10): ['Other', 'InternalMedicine', 'Family/GeneralPractice', 'Emergency/Trauma', 'Cardiology', 'Surgery-General', 'Orthopedics', 'Orthopedics-Reconstructive', 'Radiologist', 'Nephrology']
Specialties grouped into 'Other': 61
  Step 12: Group rare medical specialties into 'Other'
    Rows: 71,515 → 71,515 (Δ 0)
    Cols: 49 → 50 (Δ -1)
    Note: Kept 10 specialties, grouped rest as 'Other'



## 14. Step 13 — Data Type Corrections & Final Validation

In [17]:
# ============================================================
# Step 13: Enforce correct data types
# ============================================================

rows_b, cols_b = df.shape

# Numeric columns — ensure int/float
int_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# Categorical columns — enforce category dtype
cat_cols = [
    'race', 'gender', 'age_group', 'admission_type',
    'discharge_disposition', 'admission_source',
    'max_glu_serum', 'A1Cresult', 'readmitted',
    'readmission_status', 'diag_1_category',
    'diag_2_category', 'diag_3_category',
    'medical_specialty_grouped'
]

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

rows_a, cols_a = df.shape
log_step(13, "Enforce data types (int/category)",
         rows_b, rows_a, cols_b, cols_a,
         f"Numeric: {len(int_cols)} cols, Categorical: {len(cat_cols)} cols")

print("\nFinal data types:")
print(df.dtypes.value_counts())

  Step 13: Enforce data types (int/category)
    Rows: 71,515 → 71,515 (Δ 0)
    Cols: 50 → 50 (Δ 0)
    Note: Numeric: 8 cols, Categorical: 14 cols


Final data types:
int64       14
object      14
Int64        8
category     3
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64


In [18]:
# ============================================================
# Final validation checks
# ============================================================

print("=" * 60)
print("         FINAL VALIDATION")
print("=" * 60)

# 1. No remaining '?' values
q_remaining = (df.astype(str) == '?').sum().sum()
print(f"  Remaining '?' values:      {q_remaining} {'✅' if q_remaining == 0 else '❌'}")

# 2. No duplicate patients
dup_patients = df['patient_nbr'].duplicated().sum()
print(f"  Duplicate patients:        {dup_patients} {'✅' if dup_patients == 0 else '❌'}")

# 3. No null values in critical columns
critical_cols = ['race', 'gender', 'age_group', 'readmitted']
null_critical = df[critical_cols].isna().sum().sum()
print(f"  Nulls in critical cols:    {null_critical} {'✅' if null_critical == 0 else '❌'}")

# 4. Target is valid
valid_targets = set(['<30', '>30', 'NO'])
actual_targets = set(df['readmitted'].unique())
print(f"  Valid target values:       {actual_targets == valid_targets} {'✅' if actual_targets == valid_targets else '❌'}")

# 5. Shape summary
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("=" * 60)

         FINAL VALIDATION


  Remaining '?' values:      0 ✅
  Duplicate patients:        0 ✅
  Nulls in critical cols:    0 ✅
  Valid target values:       True ✅

  Final shape: 71,515 rows × 50 columns


## 15. Cleaning Pipeline Summary

In [19]:
# ============================================================
# Full cleaning log — every transformation documented
# ============================================================

log_df = pd.DataFrame(cleaning_log)
print("\n" + "=" * 80)
print("               COMPLETE CLEANING PIPELINE LOG")
print("=" * 80)
print(log_df[['step', 'action', 'rows_before', 'rows_after', 'rows_removed',
              'cols_before', 'cols_after', 'cols_removed']].to_string(index=False))
print("=" * 80)


               COMPLETE CLEANING PIPELINE LOG
 step                                                     action  rows_before  rows_after  rows_removed  cols_before  cols_after  cols_removed
    1                                       Replace '?' with NaN       101766      101766             0           50          50             0
    2             Drop high-missing columns (weight, payer_code)       101766      101766             0           50          48             2
    3              Drop 15 near-zero variance medication columns       101766      101766             0           48          33            15
    4                      Drop rows with Unknown/Invalid gender       101766      101763             3           33          33             0
    5          Deduplicate by patient_nbr (keep first encounter)       101763       71515         30248           33          33             0
    6 Handle missing values (medical_specialty, race, diagnoses)        71515       71515      

## 16. Export Cleaned Dataset

In [20]:
# ============================================================
# Save cleaned dataset to data/processed/
# ============================================================

# Ensure output directory exists
os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)

df.to_csv(PROCESSED_DATA_PATH, index=False)

output_size_mb = os.path.getsize(PROCESSED_DATA_PATH) / (1024 * 1024)

print(f"✅ Cleaned dataset saved to: {PROCESSED_DATA_PATH}")
print(f"   File size: {output_size_mb:.2f} MB")
print(f"   Shape:     {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n🔗 Next Step → 03_eda.ipynb")

✅ Cleaned dataset saved to: ../data/processed/cleaned_healthcare.csv
   File size: 21.35 MB
   Shape:     71,515 rows × 50 columns

🔗 Next Step → 03_eda.ipynb


## Cleaning Assumptions & Decisions

| # | Decision | Justification |
|---|----------|---------------|
| 1 | Dropped `weight` column | 96.9% missing — no imputation strategy is defensible at this level |
| 2 | Dropped `payer_code` | Not directly related to clinical readmission; 39.6% missing |
| 3 | Kept only first encounter per patient | Prevents data leakage and ensures independence of observations |
| 4 | Filled `medical_specialty` NaN with `Other` | 49% missing, but specialty is valuable in grouped form |
| 5 | Mapped `age` to midpoints | Enables quantitative analysis while preserving original groups |
| 6 | `None` in lab results → `Not Measured` | Clinically meaningful — absence of a test is informative |
| 7 | Grouped rare specialties below 1% threshold | Prevents sparsity in categorical analysis |
| 8 | ICD-9 codes → clinical categories | ~700 raw codes → ~18 interpretable groups |
| 9 | Removed `Unknown/Invalid` gender (3 rows) | Data quality — negligible sample loss |
| 10 | Dropped near-zero variance medication columns | >99% 'No' provides no analytical signal |

---

**Pipeline complete.** The cleaned dataset is now in `data/processed/cleaned_healthcare.csv`.